# पाठ 18: क्रिप्टोग्राफिक रसीदों के साथ AI एजेंट्स को सुरक्षित करना

## व्यावहारिक नोटबुक

यह नोटबुक चार अभ्यासों के माध्यम से चलता है:

1. एक एजेंट टूल कॉल के लिए **अपनी पहली रसीद पर हस्ताक्षर करें** और इसे सत्यापित करें।
2. **रसीद में छेड़छाड़ करें** और सत्यापन असफल होता देखें।
3. **तीन-रसीद श्रृंखला बनाएँ** और श्रृंखला की अखंडता की पुष्टि करें।
4. **Microsoft एजेंट फ्रेमवर्क टूल कॉल को लपेटें** ताकि प्रत्येक क्रिया एक रसीद जारी करे।

सभी क्रिप्टोग्राफिक प्रिमिटिव्स अच्छी तरह से रखरखाव की गई लाइब्रेरीज़ से आयात किए गए हैं (`pynacl` Ed25519 के लिए, `jcs` RFC 8785 कैनोनिकल JSON के लिए, Python मानक लाइब्रेरी से `hashlib` SHA-256 के लिए)। रसीद तर्क स्वयं सादा Python है जिसे आप पढ़ और संशोधित कर सकते हैं।

सेल्स को क्रम में चलाएँ। प्रत्येक अनुभाग छोटा और स्व-निहित है।


## सेटअप

दो निर्भरता स्थापित करें। दोनों के पास उदार लाइसेंस हैं (Apache-2.0 / MIT)।


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## सहायक उपयोगिताएँ

ये दो सहायक बेस64url एन्कोडिंग (पैडिंग के बिना) और मनमाने ऑब्जेक्ट्स के SHA-256 हैशिंग को संभालते हैं। ये बाकी नोटबुक को केवल रसीद लॉजिक पर केंद्रित रखते हैं।


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## अनुभाग 1: अपनी पहली रसीद पर हस्ताक्षर करें

कल्पना करें कि हमारी एजेंट **Contoso Travel** के लिए अभी एक ग्राहक के लिए सिडनी से लॉस एंजिल्स की उड़ानें देख रहा है। हम इस टूल कॉल को एक हस्ताक्षरित रसीद के रूप में रिकॉर्ड करना चाहते हैं ताकि भविष्य में कोई ऑडिटर इसे बिना हम पर भरोसा किए सत्यापित कर सके।

### चरण 1.1: एक साइनिंग कुंजी उत्पन्न करें

उत्पादन में, एजेंट की साइनिंग कुंजी एक हार्डवेयर सुरक्षा मॉड्यूल (HSM), Azure Key Vault, या समान संरक्षित स्टोर में रहती है। इस पाठ के लिए हम मेमोरी में एक नई कुंजी उत्पन्न करते हैं।


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### चरण 1.2: रसीद पेलोड बनाएँ

पेलोड में वह सब कुछ शामिल होता है जिसकी हम चाहते हैं कि रसीद प्रमाणित करे: किसने कार्य किया, कौन सा उपकरण, किन तर्कों के साथ, क्या वापस आया, किस नीति के तहत, और कब। हम तर्कों और परिणाम को इनलाइन शामिल करने के बजाय उनका हैश करते हैं ताकि रसीद संवेदनशील सामग्री को लीक न करे।


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### चरण 1.3: रसीद पर हस्ताक्षर करें और उसे एकत्रित करें

तीन चरण:

1. JCS का उपयोग करके पेलोड को कैनोनिकलाइज़ करें ताकि दो अलग-अलग कार्यान्वयन जो समान तार्किक रसीद उत्पन्न करते हैं, बाइट-समान बाइट्स उत्पन्न करें।
2. कैनोनिकल बाइट्स को SHA-256 से हैश करें।
3. Ed25519 प्राइवेट की के साथ हैश पर हस्ताक्षर करें।

फिर हस्ताक्षर को अंतिम रसीद बनाने के लिए मूल पेलोड से जोड़ा जाता है।


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### चरण 1.4: रसीद सत्यापित करें

सत्यापन प्रक्रिया को उलट देता है। हम हस्ताक्षर को हटाते हैं, पुनः कैनॉनिकल हैश की गणना करते हैं, और रसीद में सार्वजनिक कुंजी के खिलाफ हस्ताक्षर की जांच करते हैं।

इस सत्यापन को करने वाला ऑडिटर हमसे कुछ भी नहीं मांगता सिवाय रसीद के। कोई सेवा कॉल करने की आवश्यकता नहीं, कोई कुंजी निर्देशिका पूछताछ करने की जरूरत नहीं, कोई भरोसा आवश्यक नहीं।


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

आपको `Receipt is valid: True` दिखाई देना चाहिए। एजेंट ने अपना पहला क्रिप्टोग्राफिक रूप से साइन किया गया ऑडिट रिकॉर्ड बनाया है।


## अनुभाग 2: रसीद के साथ छेड़छाड़ करें

रसीदों का पूरा उद्देश्य यह है कि वे छेड़छाड़ के प्रमाण हों। आइए इसे साबित करें।

हम रसीद के ठीक एक अक्षर को बदलेंगे और सत्यापन विफल होते देखेंगे।


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### क्या अभी हुआ?

जब हमने `policy_id` बदल दिया, तो कैनोनिकल बाइट्स बदल गए। उन बाइट्स का SHA-256 हैश बदल गया। हस्ताक्षर (जो मूल हैश पर था) अब नए हैश से मेल नहीं खाता। सत्यापन सही ढंग से `False` लौटाता है।

रसीद के किसी भी फ़ील्ड को संशोधित करके उसे सत्यापित करवाना संभव नहीं है, जब तक कि हमलावर के पास निजी कुंजी न हो। जब तक निजी कुंजी एक कुंजी वाल्ट में है और सार्वजनिक कुंजी प्रकाशित है, छेड़छाड़ को छिपाना असंभव है।

इसे खुद आज़माएं: ऊपर दिए गए सेल में `tool_name` या `agent_id` या `timestamp` को संशोधित करें और फिर से चलाएं। हर बदलाव एक अमान्य रसीद उत्पन्न करता है।


## खंड 3: रसीदों को एक साथ जोड़ना

एक एकल रसीद एक क्रिया की सुरक्षा करती है। अधिकांश एजेंट कई क्रियाएँ करते हैं। पूरी श्रृंखला को छेड़छाड़-प्रमाण बनाने के लिए, हम प्रत्येक रसीद को पिछले रसीद की हैश को नई रसीद के पेलोड में शामिल करके पिछले रसीद से जोड़ते हैं।

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

यदि कोई किसी रसीद को हटाता है या पुनः क्रमबद्ध करता है, तो श्रृंखला बिल्कुल उसी बिंदु पर टूट जाती है। किसी भी बाद की रसीद का सत्यापन विफल हो जाता है क्योंकि उसकी `previous_receipt_hash` अब इसके पूर्ववर्ती की वास्तविक हैश से मेल नहीं खाती।


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

अब मध्य रसीद में छेड़छाड़ करके श्रृंखला तोड़ें और पुनः सत्यापन करें। छेड़ी गई रसीद अपनी हस्ताक्षर जांच में असफल होती है, और अगली रसीद अपनी श्रृंखला-लिंक जांच में विफल होती है (क्योंकि उसका `previous_receipt_hash` अब संशोधित मध्य रसीद के हैश से मेल नहीं खाता)।  


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

रसीद 0 अभी भी सत्यापित होती है (इसमें कोई संशोधन नहीं हुआ है और इसका कोई पूर्ववर्ती नहीं है जिस पर निर्भर किया जाए)। रसीद 1 अपनी हस्ताक्षर जांच में विफल होती है क्योंकि हमने `tool_args_hash` को बदल दिया है। रसीद 2 अपनी चेन-लिंक जांच में विफल होती है क्योंकि इसका `previous_receipt_hash` मूल (अब-संशोधित) रसीद 1 के खिलाफ गणना किया गया था।

यहां तक कि यदि कोई हमलावर संशोधित रसीद 1 पर पुनः हस्ताक्षर करता है (जो वे निजी कुंजी के बिना नहीं कर सकते), तब भी रसीद 2 में चेन-लिंक की असंगति छेड़छाड़ को उजागर कर देगी। परिवर्तन को छिपाने के लिए, हमलावर को संशोधन बिंदु से आगे की हर रसीद पर पुनः हस्ताक्षर करना होगा, जिसके लिए निजी कुंजी का होना आवश्यक है।


## खंड 4: रसीद साइनिंग के साथ एक एजेंट टूल कॉल को लपेटना

वास्तविक परिनियोजन में, आप नहीं चाहते कि हर एजेंट लेखक को `make_receipt` कॉल करना याद रहे। आप चाहते हैं कि हर टूल इनवोकेशन के लिए रसीद साइनिंग स्वचालित हो।

यहां सबसे सरल पैटर्न है: एक रैपर क्लास जो किसी भी कॉल करने योग्य टूल फ़ंक्शन को लेता है और इसका एक रसीद-प्रसारित संस्करण वापस करता है। यह किसी भी एजेंट फ्रेमवर्क के लिए उपयुक्त है, जिसमें Microsoft एजेंट फ्रेमवर्क (`agent_framework.foundry`) भी शामिल है।

यदि आपके पास Microsoft Foundry प्रोजेक्ट सेटअप नहीं है, तो नीचे का स्थानीय मॉक अभी भी इस पैटर्न को दर्शाता है।


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### Microsoft Agent Framework के साथ एकीकरण

ऊपर दिया गया `ReceiptedTool` रैपर फ्रेमवर्क-स्वतंत्र है। इसे Microsoft Agent Framework के साथ बनाए गए एजेंट के अंदर उपयोग करने के लिए, रैप की गई फ़ंक्शन को टूल के रूप में पंजीकृत करें। एक खाका (आप इसे एक वास्तविक Microsoft Foundry टूल पंजीकरण से बदलेंगे):

```python
# एकीकरण आकृति दिखाने वाला छद्मकोड।
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="आप एक Contoso यात्रा एजेंट हैं ...",
#     tools=[receipted_lookup],   # लपेटा हुआ टूल, कच्चा फ़ंक्शन नहीं
# )
# response = agent.run("जून में सिडनी से लॉस एंजिल्स के लिए उड़ानें खोजें।")
#
# # दौड़ने के बाद, एजेंट द्वारा किए गए हर टूल कॉल के लिए एक हस्ताक्षरित रसीद होती है:
# audit_chain = receipted_lookup.receipts
```

एजेंट फ्रेमवर्क को रसीदों के बारे में कुछ भी जानने की आवश्यकता नहीं है। रसीद पर हस्ताक्षर टूल के चारों ओर लपेटा गया है, फ्रेमवर्क में जोड़ा नहीं गया। यह वही तरीका है जिससे आप एजेंट को बिना पुनः लिखे उसके मौजूदा कोड में स्रोत जोड़ सकते हैं।


## पुनरावृत्ति और स्ट्रेच चुनौती

आपके पास है:

- एक Ed25519 कुंजी जोड़ी बनाई।
- एक एजेंट टूल कॉल के लिए रसीद बनाई और हस्ताक्षरित की।
- केवल सार्वजनिक कुंजी का उपयोग करके ऑफ़लाइन रसीद सत्यापित की।
- रसीद के साथ छेड़छाड़ की और सत्यापन विफलता देखी।
- तीन रसीदों की हैश-चेन की गई अनुक्रम बनायी।
- चेन के बीच में छेड़छाड़ की और दोनों हस्ताक्षर विफलता और चेन-लिंक विफलता देखी।
- एक टूल फ़ंक्शन को स्वचालित रसीद हस्ताक्षर के साथ लपेटा।

**स्ट्रेच चुनौती।** रसीद स्कीमा का विस्तार करें एक `request_id` फ़ील्ड (वितरित ट्रेसिंग के लिए UUID) के साथ। `make_receipt` को इसे शामिल करने के लिए अपडेट करें, और पुष्टि करें कि रसीदें अब भी अंत से अंत तक सत्यापित होती हैं। फिर हस्ताक्षर के बाद फ़ील्ड को संशोधित करें और पुष्टि करें कि सत्यापन विफल हो जाता है। यह आपको यह आंतरिक रूप से समझने के लिए मजबूर करता है कि मानक एनकोडिंग का हर बाइट हस्ताक्षर में कैसे योगदान देता है।

**महत्वपूर्ण सीमा।** रसीदें तीन चीजें और केवल तीन चीजें साबित करती हैं: सम्बद्धता (इस कुंजी ने इस सामग्री पर हस्ताक्षर किया), अखंडता (सामग्री हस्ताक्षर के बाद से नहीं बदली है), और क्रमबद्धता (यह रसीद उस रसीद के बाद आई)। वे यह साबित नहीं करतीं कि एजेंट की क्रिया सही थी, कि `policy_id` में नामित नीति वास्तव में मूल्यांकित हुई थी, या कि एजेंट ने हर नियम का पालन किया। रसीदें एक आधार हैं। शासन वह प्रणाली है जो आप इसके ऊपर बनाते हैं।

इस सीमा को ध्यान में रखते हुए पाठ पाठ मे पुनः देखें। रसीदों के साथ टीमों की सबसे आम गलती यह मान लेना है कि "हमारे पास रसीदें हैं" मतलब "हमें शासन मिला हुआ है।" ऐसा नहीं है। रसीदें एजेंट के व्यवहार को ऑडिटेबल बनाती हैं। वे उसे सही नहीं बनातीं।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
